# [LAB] Streamlit RAG 챗봇 실습

보험 약관을 근거로 답하는 RAG 챗봇을 웹 화면으로 띄웁니다. Streamlit 앱은 노트북 셀에서 바로 돌지 않으므로, 셀이 앱 파일을 쓰고 그다음 셀이 그 파일을 실행해 브라우저에 앱을 엽니다.

- **2장 버튼 앱으로 재실행 보기** (예제): 버튼을 누를 때마다 스크립트 전체가 다시 실행되는 모습을 봅니다. `# ✍️ CHANGE_HERE:` 줄의 늘어나는 값을 바꿔 보고, 일반 변수로 세면 어떻게 되는지 AI에게 묻습니다.
- **3장 보험 약관 QA 챗봇** (과제): 대화 기록 저장 두 곳과 답변 스트리밍 한 곳을 AI와 함께 채웁니다. 앱 제목 아래 안내 문구도 바꿔 봅니다.

앱을 띄운 셀은 멈출 때까지 끝나지 않습니다. 다음 셀로 넘어가기 전에 그 셀의 실행을 중지(■)합니다.

In [ ]:
%cd ../../../course/Chapter01/notebooks

**목차**

- 1. 실습 방법
- 2. 버튼 앱으로 재실행 보기 (예제)
  - 2.1 버튼 앱 쓰고 띄우기 (예제)
  - 2.2 AI에게 물어보기
- 3. 보험 약관 QA 챗봇 (예제) (과제)
  - 3.1 챗봇 앱 채우고 쓰기 (예제) (과제)
- 4. 바이브 코딩 과제: 역할을 고르는 Streamlit 챗봇
  - 4.1 과제 파일 만들기
  - 4.2 AI와 함께 만들고 실행하기
  - 4.3 채점받기
- 5. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env(["EMBED_PROVIDER"])

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고친 뒤 바로 위 환경 점검 셀을 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/06-rag-streamlit/test.py`·`app.py`(raw). 각 파일을 `%%writefile` 한 셀로 넣어 노트북 폴더(`lessons/Chapter01/`)에 `test.py`·`app.py`를 쓰고, 빌더가 만든 `!streamlit run` 셀로 띄웁니다. 순서는 test → app(2026-09-28 사용자 결정, 원본 이름 순과 반대).
- 노트북에서만 원본을 손봤습니다(patch, 원본 파일은 그대로): app.py의 강사 말투 주석 7줄("가우스 모델을 호출할건데…", "한개의 runnable을 받았다", "runnable을 실행하는 방법 -> invoke, stream", "화면에 보여주기 위한 session 관리", "LLM이 이전 대화를 기억한다거나 그런건 X", graph State 비유 등)을 뺐습니다. 솔루션본에도 이 주석은 없습니다. "LLM이 이전 대화를 기억하지 않는다"는 내용은 강의 쪽 session_state 절의 "왜 배우나"로 옮겼습니다.
- app.py의 약관 PDF 경로를 `Path(__file__).parent / 'data' / ...`에서 `Path("data/meritz_pet_insurance.pdf")`로 바꿨습니다. `streamlit run`을 노트북 폴더에서 실행하므로 `lessons/Chapter01/data/meritz_pet_insurance.pdf`(원본과 같은 파일)를 읽습니다.
- 생성되는 `lessons/Chapter01/app.py`·`test.py`는 Git 제외 대상이 아닙니다(`lessons/.gitignore`는 `work/`만). 강사 저장소에서 실행한 뒤에는 커밋에 섞이지 않게 지웁니다.
- `!streamlit`은 셸 PATH에서 찾습니다. 그래서 앱 셀은 `!"{sys.executable}" -m streamlit run …`으로 커널의 파이썬을 씁니다(따옴표는 경로에 공백이 있는 Windows 계정용).
- 앱 셀에 `--server.showEmailPrompt false --server.address 127.0.0.1`을 붙였습니다(2026-09-29). ① 새 PC(`~/.streamlit/credentials.toml` 없음)의 첫 `streamlit run`은 이메일 입력을 기다리며 앱이 뜨지 않습니다(재현됨). Windows의 IPython `!`는 프로세스가 끝나야 출력을 보여 줘 "Email:" 문구도 보이지 않습니다. ② 기본값(모든 인터페이스)이면 Windows 방화벽 허용 창이 뜨고, 같은 망의 다른 PC가 수강생 앱(수강생 키로 모델 호출)에 접속할 수 있습니다. 127.0.0.1이면 출력은 "Local URL/Network URL" 대신 "URL: http://127.0.0.1:8501" 한 줄입니다. `--server.headless true`는 브라우저 자동 열림도 꺼지므로 쓰지 않습니다.
- Windows에서는 ■로 셀을 멈춰도 Streamlit이 8501을 계속 잡고 있어 다음 실행이 8502로 뜰 수 있습니다(포트를 지정하지 않았으므로 Streamlit이 다음 번호로 넘어감). 안내 문구에 반영했습니다. 강의장 Windows PC에서 실제 동작은 수업 전에 확인합니다.
- 성공 기준은 자동 검사하지 않습니다(2026-09-28 결정). CodeMate 지시문 완성본은 주지 않습니다.

</details>

## 1. 실습 방법

실습은 두 종류입니다. 제목 옆 표시와 코드 주석으로 구분합니다.

- **예제** `# ✍️ CHANGE_HERE:` 바로 아래 줄의 값을 바꿔 다시 실행해 봅니다.
- **과제** `# ✍️ WRITE_HERE:` 아래 빈칸(`...`)을 AI와 함께 채웁니다. **프롬프트 힌트**를 보고 지시를 직접 씁니다.

실행한 뒤에는 **성공 기준**과 비교해 스스로 판단합니다. 🟢는 되어야 하는 것, ⛔는 다시 해야 하는 신호입니다. 과제는 그다음 접힌 "정답 보기"와 비교합니다.

"정답 보기"의 코드는 빈칸 줄과 같은 들여쓰기로 보여 줍니다. 붙여 넣을 때는 빈칸 줄 전체(줄 앞 공백까지)를 지우고 그 자리에 붙여 넣습니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. 버튼 앱으로 재실행 보기 <small>(예제)</small>

버튼 하나와 숫자 하나만 있는 작은 앱입니다. 버튼을 누르면 풍선이 뜨고 누른 횟수가 늘어납니다.

Streamlit은 버튼을 누를 때마다 스크립트를 위에서부터 **전체 다시 실행**합니다. 그래도 횟수가 유지되는 까닭은 `st.session_state`(다시 실행되어도 남는 저장 공간)에 담았기 때문입니다.

**왜 배우나:** 3장 챗봇의 대화 기록도 같은 원리로 남깁니다. 가장 작은 앱에서 재실행을 먼저 눈으로 봅니다.

### 2.1 버튼 앱 쓰고 띄우기 <small>(예제)</small>

**✍️ CHANGE_HERE: 한 번 누를 때 늘어나는 값을 바꿔 봅니다**

먼저 그대로 실행해 파일을 쓰고, 아래 셀로 앱을 띄워 버튼을 눌러 봅니다. 그다음 앱 셀을 중지(■)합니다. 이때 열려 있던 앱 탭도 멈추니 닫습니다. `# ✍️ CHANGE_HERE:` 줄의 `1`을 다른 수(예: `10`)로 바꿔 두 셀을 다시 실행합니다.

In [ ]:
%%writefile test.py
import streamlit as st

st.title("버튼 누르면 스크립트 전체 다시 실행")

if "clicks" not in st.session_state:
    st.session_state.clicks = 0

if st.button(" 클릭 "):
    # ✍️ CHANGE_HERE: 한 번 누를 때 늘어나는 값을 바꿔 봅니다
    st.session_state.clicks += 1
    st.balloons()

st.write(f"풍선 발사 {st.session_state.clicks}회")

**성공 기준**

- 🟢 셀 아래에 파일을 썼다는 한 줄(Writing … 또는 Overwriting …)이 출력되었다
- 🟢 버튼을 한 번 누를 때마다 바꾼 값만큼 숫자가 늘어난다

위 셀을 실행하면 이 폴더에 앱 파일이 만들어집니다. 아래 셀을 실행하면 잠시 뒤 브라우저에 앱이 저절로 열립니다. 열리지 않으면 브라우저 주소창에 `127.0.0.1:8501`을 넣습니다. Windows에서는 앱이 도는 동안 셀 출력이 비어 있어도 정상입니다. 다시 띄웠을 때 주소 끝이 `8502`인 탭이 열리면 그 탭을 씁니다. 앞 앱이 아직 8501을 쓰고 있다는 뜻입니다. 앱을 멈출 때는 이 셀의 실행을 중지(■)합니다. 멈춰야 다른 셀을 실행할 수 있습니다.

In [ ]:
import sys

# 첫 실행 때 이메일을 묻지 않고, 이 PC에서만 열리게 합니다
!"{sys.executable}" -m streamlit run test.py --server.showEmailPrompt false --server.address 127.0.0.1

**볼 지점**
- 제목 "버튼 누르면 스크립트 전체 다시 실행" 아래에 "클릭" 버튼과 "풍선 발사 0회"가 보입니다.
- 버튼을 누를 때마다 풍선이 뜨고 숫자가 늘어납니다. 버튼을 누를 때마다 스크립트 전체가 다시 실행되지만, 숫자는 `st.session_state`에 있어 사라지지 않습니다.
- 브라우저에서 새로고침하면 숫자가 0으로 돌아갑니다. `st.session_state`는 브라우저 연결(세션)이 이어지는 동안만 남습니다.

<details><summary>강사용 메모</summary>

- 솔루션본 `test.py`는 `clicks = 0` 일반 변수 버전이라(if 줄 주석 처리) 누를 때마다 0에서 다시 시작해 "1회"에 머뭅니다. 아래 "AI에게 물어보기"의 정답 확인용으로 쓸 수 있습니다. 수강생 화면에는 넣지 않았습니다.
- 원본 버튼 글자는 앞뒤에 공백이 있는 `' 클릭 '`입니다.

</details>

### 2.2 AI에게 물어보기

**프롬프트 힌트** 2.1에서 앱 파일을 쓰는 셀의 코드를 붙여 넣고, `st.session_state.clicks` 대신 일반 변수 `clicks = 0`으로 세면 버튼을 여러 번 눌렀을 때 숫자가 어떻게 되는지 물어봅니다. 코드는 고치지 말고 결과와 이유만 설명해 달라고 합니다.

**성공 기준**

- 🟢 "누를 때마다 0에서 다시 시작해, 한 번 누를 때 늘어나는 값(2.1에서 바꾼 수)에 머문다"고 답한다
- 🟢 이유로 "버튼을 누를 때마다 스크립트 전체가 다시 실행되어 `clicks = 0`이 다시 실행된다"를 짚는다
- ⛔ "숫자가 계속 늘어난다"고 답한다 → 버튼을 누르면 스크립트가 어디서부터 다시 실행되는지 다시 묻습니다
- ⛔ 코드를 고쳐 왔다 → 고치지 말고 설명만 해 달라고 다시 적습니다

## 3. 보험 약관 QA 챗봇 <small>(예제) (과제)</small>

메리츠 펫보험 약관(PDF)을 근거로 답하는 챗봇입니다. 코드는 두 부분입니다.

- **RAG 파이프라인 준비** `load_rag_chain()`: 약관 PDF를 읽고 조각으로 나눈 뒤 임베딩해 벡터 저장소에 넣고, 검색 → 프롬프트 → LLM → 글자만 꺼내기로 이어지는 체인을 만듭니다. `@st.cache_resource`가 붙어 있어 한 번 만든 체인을 다시 씁니다.
- **Streamlit 화면**: 제목과 안내 문구, 이전 대화 다시 그리기, 입력창, 답변 말풍선입니다.

빈칸은 세 곳입니다. 질문 저장, 답변 스트리밍, 답변 저장입니다. 세 곳을 채우고 파일을 쓴 뒤 앱을 띄워 확인합니다. 빈칸(`...`)을 채우지 않고 띄우면 오류 없이 앱이 뜨지만, 그 자리에 `Ellipsis`라는 글자가 보입니다.

시작하기 전에 2장의 버튼 앱 셀이 아직 돌고 있으면 실행을 중지(■)합니다. 앱 셀이 도는 동안에는 이 장의 셀이 실행되지 않습니다.

### 3.1 챗봇 앱 채우고 쓰기 <small>(예제) (과제)</small>

**✍️ CHANGE_HERE: 제목 아래 안내 문구를 바꿔 봅니다**

이 절은 맨 나중에 합니다. 빈칸 세 곳을 채우고 앱이 동작하는 것을 확인한 다음, 앱 셀을 중지(■)하고 `# ✍️ CHANGE_HERE:` 줄의 따옴표 안 문구를 바꿔(예: "펫보험 약관에 있는 내용만 답합니다.") 파일 쓰기 셀과 앱 셀을 다시 실행합니다.

**✍️ WRITE_HERE: 사용자 메시지를 session_state에 저장해 주세요.**

입력창에서 받은 질문(`prompt`)을 대화 기록 `st.session_state.messages`에 넣는 한 줄입니다. 위쪽 "이전 대화 표시" 코드가 이 기록을 읽어 말풍선을 다시 그립니다. 아래 성공 기준은 세 빈칸을 모두 채운 뒤 앱을 띄워 한꺼번에 확인합니다.

**프롬프트 힌트** 입력받은 `prompt`를 역할 user로 `st.session_state.messages`에 추가하는 한 줄을 요청합니다. 저장 모양은 위쪽 이전 대화 표시 코드가 읽는 `role`, `content` 키에 맞추라고 알려 줍니다.

**성공 기준**

- 🟢 두 번째 질문을 해도 첫 질문 말풍선이 화면에 남아 있다
- ⛔ 새 질문을 하면 앞의 질문 말풍선이 사라진다
- ⛔ 앱 화면에 `Ellipsis`라는 글자가 보인다(빈칸을 채우지 않은 채 파일을 썼다)

**✍️ WRITE_HERE: RAG 체인의 stream() 결과를 Streamlit에 출력해 주세요.**

RAG 체인에 질문을 넣고, 만들어지는 답변을 조각으로 받아 말풍선 안에 차례로 그리는 한 줄입니다. 다 그린 전체 답변은 변수 `response`에 담아 다음 빈칸에서 씁니다.

**프롬프트 힌트** `rag_chain.stream(prompt)`로 받은 조각을 Streamlit 화면에 차례로 그리고, 완성된 답변 문자열을 변수 `response`에 담는 한 줄을 요청합니다.

**성공 기준**

- 🟢 답변이 한 번에 나오지 않고 글자가 이어지며 나타난다
- 🟢 약관에 답이 있는 질문에는 약관 내용으로 답한다(입력창의 예시 질문처럼 약관에 없는 내용이면 없다고 답할 수 있다)
- ⛔ 답변 말풍선에 답변 대신 `Ellipsis`만 보인다
- ⛔ 답변 말풍선이 비어 있거나 `None`이 보인다

**✍️ WRITE_HERE: AI 응답을 session_state에 저장해 주세요.**

화면에 다 그린 답변(`response`)을 대화 기록에 넣는 한 줄입니다. 그래야 다음 질문 때 스크립트가 다시 실행되어도 이 답변 말풍선이 다시 그려집니다.

**프롬프트 힌트** 앞 빈칸에서 받은 `response`를 역할 assistant로 `st.session_state.messages`에 추가하는 한 줄을 요청합니다. 첫 번째 빈칸과 같은 모양으로 저장하라고 알려 줍니다.

**성공 기준**

- 🟢 다음 질문을 해도 이전 AI 답변 말풍선이 남아 있다
- ⛔ 다음 질문을 하면 이전 답변이 사라진다
- ⛔ 이전 답변 자리에 `None`이 보인다

In [ ]:
%%writefile app.py
from pathlib import Path

import streamlit as st
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PDFMinerLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

from config.embeddings import get_embeddings
from config.llm import get_llm

load_dotenv()


# ── RAG 파이프라인 초기화 (앱 시작 시 한 번만 실행) ─────────────────────────
@st.cache_resource
def load_rag_chain():
    PDF_PATH = Path("data/meritz_pet_insurance.pdf")

    loader = PDFMinerLoader(str(PDF_PATH.resolve()))
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=75)
    chunks = splitter.split_documents(documents)

    embeddings = get_embeddings()
    vectorstore = Chroma.from_documents(chunks, embedding=embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    llm = get_llm()
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "당신은 문서 기반 QA 어시스턴트입니다. 주어진 context만 근거로 답하세요.\n\ncontext:\n{context}",
            ),
            ("human", "{question}"),
        ]
    )

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain


# ── Streamlit UI ──────────────────────────────────────────────────────────────

st.title("보험 약관 QA 챗봇")
# ✍️ CHANGE_HERE: 제목 아래 안내 문구를 바꿔 봅니다
st.caption("메리츠 펫보험 약관을 기반으로 답변합니다.")

# 대화 기록을 session_state에 저장 (새로고침 전까지 유지)
if "messages" not in st.session_state:
    st.session_state.messages = []

# 이전 대화 표시
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# 사용자 입력
if prompt := st.chat_input("질문을 입력하세요 (예: 보험료는 어떻게 계산되나요?)"):
    # 사용자 메시지 표시
    # ✍️ WRITE_HERE: 사용자 메시지를 session_state에 저장해 주세요.
    ...  # ✍️ 여기를 채웁니다

    with st.chat_message("user"):
        st.write(prompt)

    # AI 응답 (스트리밍)
    with st.chat_message("assistant"):
        rag_chain = load_rag_chain()
        # ✍️ WRITE_HERE: RAG 체인의 stream() 결과를 Streamlit에 출력해 주세요.
        ...  # ✍️ 여기를 채웁니다

    # ✍️ WRITE_HERE: AI 응답을 session_state에 저장해 주세요.
    ...  # ✍️ 여기를 채웁니다

**성공 기준**

- 🟢 셀 아래에 파일을 썼다는 한 줄(Writing … 또는 Overwriting …)이 출력되었다
- 🟢 안내 문구를 바꾼 뒤 다시 띄운 앱에서 제목 아래 글자가 바뀌어 보인다

<details><summary>정답 보기</summary>

```python
    st.session_state.messages.append({"role": "user", "content": prompt})
```

```python
        response = st.write_stream(rag_chain.stream(prompt))
```

```python
    st.session_state.messages.append({"role": "assistant", "content": response})
```

</details>

위 셀을 실행하면 이 폴더에 앱 파일이 만들어집니다. 아래 셀을 실행하면 잠시 뒤 브라우저에 챗봇이 저절로 열립니다. 열리지 않으면 브라우저 주소창에 `127.0.0.1:8501`을 넣습니다. Windows에서는 앱이 도는 동안 셀 출력이 비어 있어도 정상입니다. 다시 띄웠을 때 주소 끝이 `8502`인 탭이 열리면 그 탭을 씁니다. 앞 앱이 아직 8501을 쓰고 있다는 뜻입니다. 앱을 멈출 때는 이 셀의 실행을 중지(■)합니다.

In [ ]:
import sys

# 첫 실행 때 이메일을 묻지 않고, 이 PC에서만 열리게 합니다
!"{sys.executable}" -m streamlit run app.py --server.showEmailPrompt false --server.address 127.0.0.1

**볼 지점**
- 첫 질문은 답이 나오기까지 오래 걸립니다. 이때 약관을 읽고 임베딩해 체인을 만듭니다. 두 번째 질문부터는 만들어 둔 체인을 다시 써서 빨리 시작합니다.
- 답변이 한 번에 나오지 않고 글자가 이어지며 나타납니다.
- 펫보험 약관에 관한 질문을 하고, 답변이 약관 내용을 근거로 하는지 봅니다. 약관에 없는 내용을 물으면 "나와 있지 않다"는 답이 나올 수 있습니다. 주어진 context만 근거로 답하라는 지시 때문입니다. 입력창의 예시 질문(보험료 계산)이 그런 경우입니다. 약관에 답이 있는 질문(예: "자기부담금은 얼마인가요?")도 해 보고 두 답을 비교합니다.
- 질문을 두 번 이상 해 봅니다. 앞의 질문과 답변이 화면에 그대로 남아 있어야 합니다.
- 앱 코드에서 오류가 나면 셀 출력이 아니라 브라우저 화면에 빨간 상자로 보입니다. 그 글을 AI에게 붙여 넣고 원인을 물어봅니다.
- 앱을 띄운 셀의 출력에 경고 문구가 보일 수 있습니다. 앱이 동작하면 넘어갑니다.

<details><summary>강사용 메모</summary>

- `load_rag_chain()`은 원본 구분선 주석("앱 시작 시 한 번만 실행")과 달리 첫 질문 때 처음 실행됩니다(`if prompt := ...` 안에서 호출). 그래서 첫 답변만 느립니다.
- 서버를 새로 띄울 때마다 첫 질문에서 약관 전체를 임베딩합니다(메모리 Chroma, 저장 안 함). 사외 openai 경로는 소액 과금이 됩니다.
- 체인에는 이번 질문(`prompt`)만 들어갑니다. `session_state.messages`는 화면 표시용이라 "그건 얼마야?" 같은 이어지는 질문은 앞 대화를 모르고 답합니다. 수강생이 물으면 강의의 session_state "왜 배우나"로 연결합니다.
- 빈칸을 채우지 않아도 `...`은 올바른 파이썬이라 앱이 뜹니다. Streamlit의 magic(단독 표현식을 화면에 출력)이 `...`을 `Ellipsis`라는 글자로 그립니다(2026-09-28 AppTest로 확인: 답변 말풍선 안과 대화 아래에 `Ellipsis`, 대화 기록은 쌓이지 않음). 오류가 나지 않으니 ⛔ 기준으로 확인시킵니다.
- 앱을 띄운 채 파일만 다시 쓰는 것은 커널이 앱 셀에 묶여 있어 할 수 없습니다. 항상 ■ → 파일 쓰기 → 앱 실행 순서입니다. 이때 서버가 새로 떠서 캐시도 비워집니다.
- 2026-09-28 사외(openai) 실행 확인: 입력창 예시 질문 "보험료는 어떻게 계산되나요?"에는 "보험료 계산 방식이나 산출 기준이 나와 있지 않다"는 취지로 답했습니다(검색된 조각에 계산식이 없음). 시연 질문은 약관에 답이 있는 것으로 미리 골라 둡니다. 사내 모델의 답은 미확인입니다.
- 볼 지점의 예시 "자기부담금은 얼마인가요?"는 약관 PDF 글에 답이 있습니다(통원·입원의료비 1일당 1만원/2만원/3만원, 배상책임 사고당 3만원, 2026-09-29 PDF 글 추출로 확인). 검색이 그 조각을 실제로 찾는지는 모델로 실행해 보지 않았습니다. 입력창 예시(원본 코드)는 슬라이드와 같게 그대로 두었습니다.

</details>

## 4. 바이브 코딩 과제: 역할을 고르는 Streamlit 챗봇

위에서 배운 것을 조합해 새 프로그램을 AI와 함께 만듭니다. 노트북이 아니라 따로 만든 과제 파일 `2-3_task.py`에 작성하고, 터미널에서 실행합니다.

**목표** 사이드바에서 답변 역할을 고르는 Streamlit 챗봇을 만듭니다. 대화 기록은 `st.session_state`에 쌓아 화면에 남기고, 답은 글자가 이어지며 나오게 하고, 사이드바 버튼으로 대화를 지웁니다.

**들어갈 것**

- 사이드바 `st.selectbox`로 역할("친절한 선생님", "간결한 비서")을 고르고, 고른 역할을 system 메시지로 넣습니다
- 대화 기록을 `st.session_state`에 쌓고, 화면을 다시 그릴 때마다 `st.chat_message`로 모두 보여 줍니다
- 질문은 `st.chat_input`으로 받고, 답은 `llm.stream()`을 `st.write_stream`으로 흘려 보여 준 뒤 기록에 저장합니다
- 사이드바 "대화 지우기" 버튼을 누르면 기록을 비웁니다

**성공 기준**

- 🟢 질문을 두 번 하면 두 질문과 두 답이 모두 화면에 남는다
- 🟢 답이 고른 역할의 말투로 나온다
- 🟢 "대화 지우기"를 누르면 기록이 사라진다

**프롬프트 힌트** 목표와 들어갈 것을 그대로 AI에게 주고, 과제 파일에만 작성해 달라고 합니다. 앱은 터미널에서 띄우고, 고친 뒤에는 브라우저에서 새로 고침하면 된다고 알려 줍니다.

### 4.1 과제 파일 만들기

아래 셀을 실행하면 이 노트북 옆에 `2-3_task.py` 파일이 만들어집니다. 모델을 불러오는 줄은 미리 들어 있습니다. 파일이 이미 있으면 덮어쓰지 않으니 다시 실행해도 작업한 내용이 지워지지 않습니다.

In [ ]:
from pathlib import Path

task_file = Path("2-3_task.py")
starter = '''\
"""2-3 과제: 역할을 고르는 Streamlit 챗봇"""

import streamlit as st
from dotenv import load_dotenv

from config.llm import get_llm

load_dotenv()

llm = get_llm()

st.title("역할 챗봇")


# 아래에 AI와 함께 코드를 작성합니다
'''

# 이미 있으면 덮어쓰지 않습니다(작업한 내용이 지워지지 않게)
if task_file.exists():
    print(f"이미 있습니다. 그대로 둡니다: {task_file.resolve()}")
else:
    task_file.write_text(starter, encoding="utf-8")
    print(f"만들었습니다: {task_file.resolve()}")

### 4.2 AI와 함께 만들고 실행하기

1. VS Code 탐색기에서 `course/Chapter01/notebooks/2-3_task.py`를 엽니다.
2. AI에게 목표와 들어갈 것을 주고 이 파일에 작성해 달라고 합니다. AI가 만든 코드를 읽어 보고, 성공 기준과 다르면 다시 지시합니다.
3. VS Code 메뉴 **터미널 → 새 터미널**을 열고(저장소 맨 위 폴더에서 열립니다) 아래 명령으로 실행합니다.

```
uv run --env-file .env streamlit run course/Chapter01/notebooks/2-3_task.py
```

4. 브라우저에 앱이 열립니다. 질문해 보고 성공 기준과 비교합니다. 파일을 고쳤으면 브라우저에서 새로 고침합니다. 앱을 끌 때는 터미널에서 **Ctrl + C**를 누릅니다.

### 4.3 채점받기

다 됐으면 AI에게 **"2-3 과제 채점해 줘"**라고 말합니다. AI가 과제 파일을 직접 실행해 성공 기준마다 🟢 통과 / ⛔ 수정 필요 / ❔ 확인 불가를 알려 주고, 다음에 할 일 하나와 확인 질문 하나를 냅니다.

## 5. 정리

**이 노트북에서 한 것**
- 셀로 앱 파일을 쓰고, 그 파일을 실행해 브라우저에 Streamlit 앱을 띄웠습니다.
- 버튼을 누르거나 질문을 입력할 때마다 스크립트 전체가 다시 실행되는 것을 보았습니다. 남겨야 하는 값은 `st.session_state`에 담았습니다.
- 질문과 답변을 대화 기록에 저장해, 다시 실행되어도 이전 대화가 화면에 남게 했습니다.
- RAG 체인의 답변을 `st.write_stream`으로 글자가 이어지며 나타나게 했습니다.
- 무거운 체인 준비는 `@st.cache_resource`로 한 번만 하고 다시 썼습니다.

마치기 전에 앱 셀이 아직 돌고 있으면 실행을 중지(■)하고 앱 브라우저 탭을 닫습니다.